# Notebook 04b: LeadwiseTransformer — Cross-Lead ECG Classification

**Novel contribution of this project.** Every other model in this pipeline treats the 12 ECG leads as independent channels (channel-independent). `LeadwiseTransformer` instead models explicit *inter-lead relationships* — the architectural insight that mirrors clinical practice:

> A cardiologist diagnosing inferior MI does not read lead II in isolation — they observe concordant ST elevation across II, III, and aVF simultaneously.

## Architecture

```
Input (B, 12, 1000)
  |
  +-- Conv1d patch embedding (per lead, shared weights)
  |
  +-- Temporal Transformer encoder  (3 layers, Pre-LN, shared across leads)
  |     Captures within-lead temporal patterns
  |
  +-- Cross-lead Multi-head Attention  <-- novel part
  |     Q = K = V = lead token sequence
  |     Each lead attends to all other leads
  |
  +-- Mean pool over leads
  |
  +-- Classifier head  -> (B, 5) logits
```

Total parameters: ~476 K, all trained from scratch.

Results are compared against HuBERT baselines in `04c_comparison.ipynb`.

In [1]:
import sys, os, warnings
sys.path.append('../')
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

import torch
from src.utils.config                import CFG
from src.preprocessing.label_utils   import load_all_labels
from src.preprocessing.dataset_full  import ECGDatasetFull
from src.models.leadwise_transformer import LeadwiseTransformer
from src.training.train_peft         import run_experiment

DATA_PATH = CFG['preprocessing']['path']
device    = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'GPU:   {torch.cuda.get_device_name(0)}')
print(f'VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Results dir: {CFG["paths"]["results"]}')

GPU:   NVIDIA GeForce RTX 4060 Laptop GPU
VRAM:  8.6 GB
Results dir: D:\GitHub\biosignal-xai\results/


## Data

Same PTB-XL split used in notebook 04: folds 1–8 for training, fold 9 for validation. `ECGDatasetFull` is used — one record per sample, shape `(12, 1000)`. The test fold (10) remains held out.

In [2]:
Y = load_all_labels(
    DATA_PATH + 'ptbxl_database.csv',
    DATA_PATH + 'scp_statements.csv',
)
train_df = Y[Y.strat_fold <  9]
val_df   = Y[Y.strat_fold == 9]

train_ds = ECGDatasetFull(train_df, DATA_PATH)
val_ds   = ECGDatasetFull(val_df,   DATA_PATH)

print(f'Train: {len(train_df):,} records -> {len(train_ds):,} samples')
print(f'Val:   {len(val_df):,} records  -> {len(val_ds):,} samples')

Records with valid labels: 21388
Class distribution:
  NORM: 9514 (44.5%)
  MI: 5469 (25.6%)
  STTC: 5235 (24.5%)
  CD: 4898 (22.9%)
  HYP: 2649 (12.4%)
Train: 17,084 records -> 17,084 samples
Val:   2,146 records  -> 2,146 samples


## Architecture Verification

Before training, verify that `(B, 12, 1000)` input produces `(B, 5)` logits and confirm parameter count.

In [3]:
_dummy = torch.randn(4, 12, 1000)
model  = LeadwiseTransformer()
p      = model.count_parameters()
with torch.no_grad():
    out = model(_dummy)
assert out.shape == (4, 5), f'Expected (4, 5), got {out.shape}'
print(f'LeadwiseTransformer')
print(f'  Trainable: {p["trainable"]:,} / {p["total"]:,} ({p["percentage"]})')
print(f'  Forward:   {tuple(_dummy.shape)} -> {tuple(out.shape)} -- OK')
del model, out, _dummy

LeadwiseTransformer | Trainable: 476,037 / 476,037 (100.0%)
LeadwiseTransformer
  Trainable: 476,037 / 476,037 (100.0%)
  Forward:   (4, 12, 1000) -> (4, 5) -- OK


## Training: LeadwiseTransformer — Full Parameter Optimization

All 476 K parameters are trained from scratch. Because there is no pretrained backbone to preserve, every weight must be learned from PTB-XL data alone. This uses the same `run_experiment` loop as notebook 04: warmup LR schedule, BCEWithLogitsLoss with class-imbalance `pos_weight`, and patience-based early stopping.

**Why not LoRA?** LoRA freezes the base model and trains only small rank-decomposition deltas — the assumption being that the frozen backbone already encodes useful representations. That holds for HuBERT-ECG (pretrained on audio then ECG). Applying LoRA to a randomly initialized model freezes 459 K weights at random initialization; the 17 K adapter deltas cannot recover discriminative features from a permanently random computation graph, and the model collapses to predicting all-negative (AUC ~0.50, F1 = 0).

In [4]:
model = LeadwiseTransformer()

auc, hist, _ = run_experiment(
    model, train_ds, val_ds,
    experiment_name='leadwise_transformer',
    epochs=CFG['training']['epochs'],
    lr=CFG['training']['lr_peft'],
    batch_size=CFG['training']['batch_size_full'],
    save_dir=CFG['paths']['results'],
)
del model; torch.cuda.empty_cache()

print(f'\nLeadwise (full training): AUC {auc:.4f}')
print(f'Full comparison and plots -> 04c_comparison.ipynb')

print()
print('NOTE -- why LoRA does not work for LeadwiseTransformer:')
print('  LoRA freezes the base model weights and trains only small low-rank adapter deltas.')
print('  This makes sense for pretrained models (e.g. HuBERT-ECG): the frozen backbone')
print('  already encodes useful representations that adapters fine-tune.')
print('  LeadwiseTransformer has no pretrained weights -- applying LoRA freezes 459 K')
print('  parameters at random initialization. The 17 K adapter deltas cannot recover')
print('  discriminative features from a permanently random backbone, so the model')
print('  converges to predicting all-negative (F1 = 0, AUC ~ 0.50 = random chance).')

LeadwiseTransformer | Trainable: 476,037 / 476,037 (100.0%)

 Experiment : leadwise_transformer
 Device     : cuda
 Trainable  : 476,037  (100.0%)

Epoch 01/15  lr=1.50e-04
  train_loss=0.8782  val_loss=0.8483
  AUC (macro): 0.4919
  F1  (macro): 0.2268
  Per-class AUC:
    NORM : 0.473  #########
    MI   : 0.506  ##########
    STTC : 0.495  #########
    CD   : 0.503  ##########
    HYP  : 0.483  #########
Saved -> D:\GitHub\biosignal-xai\results/leadwise_transformer\best_adapter/checkpoint.pt
  * Best saved — AUC 0.4919

Epoch 02/15  lr=3.00e-04
  train_loss=0.7866  val_loss=0.7726
  AUC (macro): 0.6489
  F1  (macro): 0.0000
  Per-class AUC:
    NORM : 0.685  #############
    MI   : 0.710  ##############
    STTC : 0.588  ###########
    CD   : 0.704  ##############
    HYP  : 0.558  ###########
Saved -> D:\GitHub\biosignal-xai\results/leadwise_transformer\best_adapter/checkpoint.pt
  * Best saved — AUC 0.6489

Epoch 03/15  lr=2.96e-04
  train_loss=0.7108  val_loss=0.6394
  AUC (m